Install 

In [0]:
 
 %pip install openmeteo-requests
 %pip install requests-cache retry-requests numpy pandas

In [0]:
 %restart_python

Usage

In [0]:
# Create the daily weather table with the correct schema
spark.sql("""
CREATE TABLE IF NOT EXISTS weather_openmeteo.bronze.weather_daily (
    date TIMESTAMP,
    latitude DOUBLE,
    longitude DOUBLE,
    temperature_2m_max FLOAT,
    temperature_2m_min FLOAT,
    daylight_duration FLOAT,
    wind_speed_10m_max FLOAT
)
USING DELTA
""")

print("Table weather_openmeteo.bronze.weather_daily created successfully!")

In [0]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
import logging

# --- Spark & Delta Setup ---
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Initialize SparkSession configured for Delta Lake
spark = SparkSession.builder \
    .appName("OpenMeteoMerge") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Target table name in the data catalog
table_name = "weather_openmeteo.bronze.weather_daily"

# --- Input Validation Functions ---
def validate_coordinate(lat, lon):
    """Validate latitude and longitude values."""
    if not (-90 <= lat <= 90):
        raise ValueError(f"Invalid latitude: {lat}. Must be between -90 and 90.")
    if not (-180 <= lon <= 180):
        raise ValueError(f"Invalid longitude: {lon}. Must be between -180 and 180.")
    return True

def validate_response(response):
    """Validate API response has expected data structure."""
    if not response:
        return False
    try:
        daily = response.Daily()
        if not daily:
            return False
        # Check that we have the expected number of variables
        if daily.Variables(0) is None:
            return False
        return True
    except Exception as e:
        logger.warning(f"Response validation failed: {str(e)}")
        return False

# --- API Client Setup ---
cache_session = requests_cache.CachedSession('/tmp/.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# --- Configuration ---
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": [42.6828, 42.5084, 43.0731],
    "longitude": [-89.0187, -89.0318, -89.4012],
    "daily": ["temperature_2m_max", "temperature_2m_min", "daylight_duration", "wind_speed_10m_max"],
    "models": "gfs_seamless",
    "timezone": "auto",
    "past_days": 1,
    "temperature_unit": "fahrenheit",
}

# Validate input coordinates
try:
    for lat, lon in zip(params["latitude"], params["longitude"]):
        validate_coordinate(lat, lon)
    logger.info("Input coordinates validated successfully")
except ValueError as e:
    logger.error(f"Coordinate validation failed: {str(e)}")
    raise

# Make API request with error handling
try:
    logger.info(f"Requesting weather data from {url}")
    responses = openmeteo.weather_api(url, params=params)
    logger.info(f"Received {len(responses)} location responses")
except Exception as e:
    logger.error(f"API request failed: {str(e)}")
    raise Exception(f"Failed to fetch weather data: {str(e)}")

# Verify we got responses
if not responses or len(responses) == 0:
    raise Exception("No responses received from API")

# Instantiate the DeltaTable object
try:
    delta_table = DeltaTable.forName(spark, table_name)
    logger.info(f"Connected to table: {table_name}")
except Exception as e:
    logger.error(f"Failed to connect to table {table_name}: {str(e)}")
    raise

# Process each location with error handling
successful_merges = 0
failed_locations = []

for idx, response in enumerate(responses):
    try:
        # Validate response structure
        if not validate_response(response):
            logger.warning(f"Invalid response for location {idx+1}, skipping")
            failed_locations.append(idx+1)
            continue
        
        lat = round(response.Latitude(), 4)
        lon = round(response.Longitude(), 4)
        logger.info(f"Processing location {idx+1}: ({lat}, {lon})")
        
        # Process daily data
        daily = response.Daily()
        daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
        daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
        daily_daylight_duration = daily.Variables(2).ValuesAsNumpy()
        daily_wind_speed_10m_max = daily.Variables(3).ValuesAsNumpy()
        
        # Validate data arrays are not empty
        if len(daily_temperature_2m_max) == 0:
            logger.warning(f"No data returned for location {idx+1}, skipping")
            failed_locations.append(idx+1)
            continue
        
        # Create the Pandas DataFrame
        daily_data = {
            "date": pd.date_range(
                start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
                end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
                freq = pd.Timedelta(seconds = daily.Interval()),
                inclusive = "left"
            ).tz_convert(response.Timezone().decode())
        }
        daily_data["latitude"] = lat
        daily_data["longitude"] = lon
        daily_data["temperature_2m_max"] = daily_temperature_2m_max
        daily_data["temperature_2m_min"] = daily_temperature_2m_min
        daily_data["daylight_duration"] = daily_daylight_duration
        daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
        
        df_Weather = pd.DataFrame(data=daily_data)
        
        # Validate DataFrame is not empty
        if df_Weather.empty:
            logger.warning(f"Empty DataFrame for location {idx+1}, skipping")
            failed_locations.append(idx+1)
            continue
        
        # Convert Pandas DataFrame to Spark DataFrame
        df_spark_source = spark.createDataFrame(df_Weather)
        
        # Execute incremental Merge
        (
            delta_table.alias("target")
            .merge(
                df_spark_source.alias("source"),
                "target.date = source.date AND target.latitude = source.latitude AND target.longitude = source.longitude"
            )
            .whenMatchedUpdate(set={
                "temperature_2m_max": "source.temperature_2m_max",
                "temperature_2m_min": "source.temperature_2m_min",
                "daylight_duration": "source.daylight_duration",
                "wind_speed_10m_max": "source.wind_speed_10m_max"
            })
            .whenNotMatchedInsert(values={
                "date": "source.date",
                "latitude": "source.latitude",
                "longitude": "source.longitude",
                "temperature_2m_max": "source.temperature_2m_max",
                "temperature_2m_min": "source.temperature_2m_min",
                "daylight_duration": "source.daylight_duration",
                "wind_speed_10m_max": "source.wind_speed_10m_max"
            })
            .execute()
        )
        
        successful_merges += 1
        logger.info(f"Successfully merged data for location {idx+1}")
        
    except Exception as e:
        logger.error(f"Error processing location {idx+1}: {str(e)}")
        failed_locations.append(idx+1)
        continue

# Final summary
print(f"\n{'='*60}")
print(f"Processing completed!")
print(f"Successful merges: {successful_merges}/{len(responses)}")
if failed_locations:
    print(f"Failed locations: {failed_locations}")
else:
    print("All locations processed successfully!")
print(f"{'='*60}")

## Storing API Secrets Securely

If you upgrade to a paid API tier that requires authentication, **never hardcode API keys in your notebook**. Use Databricks Secrets instead.

### Step 1: Create a Secret Scope

Using Databricks CLI:
```bash
databricks secrets create-scope --scope api-secrets
```

Or via the UI: `https://<databricks-instance>/#secrets/createScope`

### Step 2: Store Your API Key

```bash
databricks secrets put --scope api-secrets --key openmeteo-api-key
```

This will open an editor where you can paste your API key securely.

### Step 3: Access Secrets in Your Code

```python
# Retrieve the API key from the secret scope
api_key = dbutils.secrets.get(scope="api-secrets", key="openmeteo-api-key")

# Add to API request parameters
params["apikey"] = api_key
```

### Benefits:
* **Secure Storage**: Keys are encrypted at rest and in transit
* **Access Control**: Fine-grained permissions via ACLs
* **Audit Trail**: Track who accesses which secrets
* **No Git Exposure**: Secrets never appear in notebooks or version control
* **Redacted Output**: Secret values are automatically redacted in notebook outputs

### Best Practices:
* Create separate scopes for different environments (dev, staging, prod)
* Use descriptive key names: `service-name-purpose` (e.g., `openmeteo-api-key`)
* Rotate secrets regularly
* Grant minimal necessary permissions to secret scopes
* Never print or log secret values directly